# UA-DETRAC camera calibration and explicit review

Compare direct ground-plane homography and camera K/R/T. A lane width, focal length or other dimension is an assumption until supported by scene evidence. Camera sharing requires explicit confirmation; the paper states 24 locations, not 24 known camera geometries.


## Environment

Set UA_DETRAC_PROJECT, UA_DETRAC_DATASET, UA_DETRAC_DETECTIONS and UA_DETRAC_CALIBRATION to override paths. Use local Colab storage for repeated image reads.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import json

# Mount Drive on Colab; local kernels find the repository from parent folders.
try:
    from google.colab import drive, output as colab_output
except ImportError:
    pass
else:
    drive.mount('/content/drive')
    colab_output.enable_custom_widget_manager()

project = os.environ.get('UA_DETRAC_PROJECT')
candidates = ([Path(project)] if project else [Path.cwd(), *Path.cwd().parents,
    Path('/content/drive/MyDrive/CS331/CS331-Vehicle-Tracking')])
ROOT = next((p.resolve() for p in candidates
             if (p / 'scripts/tracking_pipeline.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Set UA_DETRAC_PROJECT to the repository directory.')
sys.path.insert(0, str(ROOT))
local_dataset = Path('/content/ua_detrac/dataset_v1.1')
default_dataset = local_dataset if local_dataset.is_dir() else ROOT / 'outputs/ua_detrac/dataset_v1.1'
DATASET = Path(os.environ.get('UA_DETRAC_DATASET', os.environ.get('UA_DETRAC_OUTPUT', default_dataset)))
DETECTIONS = Path(os.environ.get('UA_DETRAC_DETECTIONS', ROOT / 'outputs/detections_full_yolo26l_E17'))
CALIBRATION = Path(os.environ.get('UA_DETRAC_CALIBRATION', ROOT / 'outputs/tracking/calibration/registry.json'))
RUNS = ROOT / 'outputs/tracking'
METHOD = 'homography'


def run_command(action, output=None, method=None, sequences=None, resume=False, extra=()):
    command = [sys.executable, str(ROOT / 'scripts/tracking_pipeline.py'), action,
        '--dataset', str(DATASET), '--detections', str(DETECTIONS),
        '--calibration', str(CALIBRATION), '--upstream', str(ROOT / 'UCMC/UCMCTrack')]
    if output is not None:
        command += ['--output', str(output)]
    if method is not None:
        command += ['--method', method]
    if sequences:
        command += ['--sequences', *sequences]
    if resume:
        command += ['--resume']
    command += list(extra)
    print('Command:', command)
    return subprocess.run(command, cwd=ROOT, check=True)

print('Dataset:', DATASET)
print('Detections:', DETECTIONS)
print('Calibration:', CALIBRATION)

# Apply the compatibility patch after initializing the pinned dependency.
subprocess.run([sys.executable, str(ROOT / 'scripts/setup_ucmc.py')], cwd=ROOT, check=True)


## Dependencies

Skip the next cell when using the prepared tracking environment.


In [ ]:
# Run once in a new environment. Local users should select the .venv-tracking kernel.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                str(ROOT / 'requirements-tracking.txt')], cwd=ROOT, check=True)


## Inventory and contact sheets

This reads prepared images and creates first/middle/last contact sheets, a master sheet and three similarity suggestions per sequence. Existing camera reviews are preserved. New cameras have needs_review status and no invented matrices.


In [ ]:
from src.tracking.calibration import prepare_registry, read_registry
from IPython.display import Image as NotebookImage, display
summary = prepare_registry(DATASET, CALIBRATION.parent)
print(json.dumps(summary, indent=2))
display(NotebookImage(filename=str(CALIBRATION.parent / 'all_sequences.jpg')))


## Fit and review geometry

The widget uses a Vietnamese five-step interface: select video/frame, click four corners, enter dimensions, preview, then save. The blue edge 1->2 is width; the orange edge 2->3 is length. Select points around the boundary, without crossing edges. Use the undo/reset buttons in step 2. The coordinate editor, KRT controls and camera-group controls are collapsed under advanced sections.

Start with MVI_20012, MVI_39811 and MVI_63521. To review an existing proposal, use the load button in step 1. Inspect all three preview frames before approving; changing geometry requires a new preview. Width 3.5 m is only a prior and length must be entered independently.

Draft proposals are in outputs/tracking/calibration/proposals_24_v1/review.html. These are representative views with unmeasured dimensions, not official camera IDs. See docs/TRACKING_USAGE_GUIDE.md for the Vietnamese walkthrough and explanations of src/tracking.


In [ ]:
import importlib
from src.tracking import review

# Reload after updating the module without restarting the notebook kernel.
importlib.reload(review)
SEQUENCE = 'MVI_20012'
display(review.calibration_widget(DATASET, CALIBRATION, sequence=SEQUENCE,
    output=CALIBRATION.parent / 'previews', detections=DETECTIONS))


## Share a camera group

Expand the advanced camera-sharing section only when needed. Publish a reviewed reference, then verify each member has matching geometry, crop, zoom and resolution before confirming assignment. A sequence-specific calibration takes precedence over a group.


## Review status

Pending cameras block normal tracking. Complete both methods on the smoke sequences, compare them, then review and freeze the chosen method on the complete validation split. Continue review on the remaining sequences for the 100-sequence deliverable.


In [ ]:
from src.tracking.calibration import resolve_calibration
inventory = read_registry(CALIBRATION)['sequences']
for method in ['homography', 'krt']:
    ready, pending = [], []
    for sequence in inventory:
        try:
            resolve_calibration(CALIBRATION, sequence, method)
        except (ValueError, KeyError) as error:
            pending.append(sequence)
        else:
            ready.append(sequence)
    print(method, 'reviewed:', len(ready), '/', len(inventory), 'pending:', pending)


## Optional upstream KRT text export

JSON uses metres internally. The upstream text reader expects translation vectors in millimetres. A homography alone cannot supply unique K/R/T.


In [ ]:
from src.tracking.calibration import export_krt
try:
    camera = resolve_calibration(CALIBRATION, SEQUENCE, 'krt')
except (ValueError, KeyError) as error:
    print('KRT export is pending:', error)
else:
    destination = CALIBRATION.parent / 'cam_para' / f'{SEQUENCE}.txt'
    export_krt(camera, destination)
    print(destination)
